# 03 — Demo: Định giá nhà Hà Nội

Giao diện web (Gradio) dùng các mô hình đã huấn luyện ở notebook 02.

| Tab | Chức năng |
|---|---|
| **Định giá** | Nhập đặc điểm nhà (hoặc dán nguyên văn tin rao để tự điền) → giá dự đoán, khoảng giá tham khảo, so sánh 7 mô hình, giải thích từng yếu tố đóng góp bao nhiêu %, thử "nếu… thì sao", vị trí trên bản đồ |
| **Kiểm tra với tin thật** | Lấy ngẫu nhiên một tin trong tập test, so giá rao thật với giá dự đoán |
| **Bản đồ giá** | Đơn giá trung vị theo quận/huyện |
| **Kết quả mô hình** | Bảng chỉ số và các biểu đồ từ notebook 02 |

**Cần có trên Drive** (`MyDrive/ML_house_price/`): `models/` (7 file `.joblib` + `demo_meta.json`), `data_processed/train.parquet`, `test.parquet`, `figures/02/`.

Chạy lần lượt các ô; ô cuối in ra **link công khai** (`…gradio.live`, hiệu lực 72 giờ) để mở trên trình duyệt hoặc gửi cho người khác. Giữ tab Colab mở trong lúc dùng demo.

In [ ]:
%pip -q install "gradio>=5,<6"

## 1. Nạp mô hình & dữ liệu

In [ ]:
import os, re, json, math, unicodedata, warnings, glob
import numpy as np
import pandas as pd
import joblib
import sklearn

warnings.filterwarnings("ignore", category=UserWarning)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

def find_project_dir():
    # Thứ tự ưu tiên: (1) biến môi trường ML_PROJECT_DIR; (2) Colab: thư mục dự án trên Google Drive;
    # (3) chạy trên máy: thư mục gốc dự án (nơi có requirements.txt) — thư mục hiện tại hoặc thư mục cha
    if os.environ.get("ML_PROJECT_DIR"):
        return os.environ["ML_PROJECT_DIR"]
    if IN_COLAB:
        return "/content/drive/MyDrive/ML_house_price"
    here = os.path.abspath(os.getcwd())
    for p in (here, os.path.dirname(here)):
        if os.path.isfile(os.path.join(p, "requirements.txt")):
            return p
    return here

PROJECT_DIR = find_project_dir()
MODEL_DIR = f"{PROJECT_DIR}/models"
DATA_DIR  = f"{PROJECT_DIR}/data_processed"
FIG_DIR   = f"{PROJECT_DIR}/figures/02"
FIG_DIR4  = f"{PROJECT_DIR}/figures/04"

META = json.load(open(os.path.join(MODEL_DIR, "demo_meta.json"), encoding="utf-8"))
if META["sklearn_version"] != sklearn.__version__:
    print(f"⚠ Mô hình huấn luyện bằng scikit-learn {META['sklearn_version']}, môi trường hiện tại là "
          f"{sklearn.__version__}. Nếu nạp lỗi, chạy: %pip install scikit-learn=={META['sklearn_version']} rồi khởi động lại runtime.")

MODELS = {name: joblib.load(os.path.join(MODEL_DIR, f)) for name, f in META["model_files"].items()}
LINEAR = [n for n, m in MODELS.items() if hasattr(m.regressor_.named_steps["reg"], "coef_")]   # mô hình có hệ số → giải thích được
NUM, BIN, CAT = META["features"]["num"], META["features"]["bin"], META["features"]["cat"]
FEATURES = NUM + BIN + CAT
FLAG_LABELS = META["binary_flags"]
print("Đã nạp:", ", ".join(MODELS))

def prepare(df):
    df = df.copy()
    for c in ["district_key", "ward_new", "house_direction", "road_type"]:
        df[c] = df[c].fillna("unknown").astype(str).replace("", "unknown")
    df["ward_loc"] = df["district_key"] + " | " + df["ward_new"]
    for c in BIN:
        df[c] = df[c].fillna(0).astype(int)
    return df

TRAIN = prepare(pd.read_parquet(os.path.join(DATA_DIR, "train.parquet")))
TEST  = prepare(pd.read_parquet(os.path.join(DATA_DIR, "test.parquet")))
print(f"Train {len(TRAIN):,} | Test {len(TEST):,}")

### Thống kê dùng trong demo
- **Căn tham chiếu để giải thích**: kích thước và vị trí bằng trung bình tập train, **chưa có tiện ích nào**. Mỗi yếu tố được giải thích bằng % chênh lệch so với căn tham chiếu (vì mô hình học log(giá), các tác động nhân với nhau).
- **Khoảng giá tham khảo**: phân vị của sai số log(giá thực) − log(giá dự đoán) trên tập test cho từng mô hình → khoảng 50% và 80%.
- **Bản đồ**: đơn giá trung vị và số tin theo quận/huyện (từ tập train); tọa độ là **tâm gần đúng** của từng quận/huyện (dữ liệu không có tọa độ từng căn).

In [ ]:
rng = np.random.default_rng(0)
_sample = TRAIN.iloc[rng.choice(len(TRAIN), size=min(50_000, len(TRAIN)), replace=False)][FEATURES]
ZBAR = {}
for n in LINEAR:
    pre = MODELS[n].regressor_.named_steps["pre"]
    ZBAR[n] = np.asarray(pre.transform(_sample)).mean(axis=0)

y_test = TEST["price"].to_numpy()
RESID_Q = {}
for n, m in MODELS.items():
    r = np.log(y_test) - np.log(np.clip(m.predict(TEST[FEATURES]), 1, None))
    RESID_Q[n] = {q: float(np.quantile(r, q)) for q in (0.10, 0.25, 0.75, 0.90)}
print("Khoảng sai số log (50%) —", {n: (round(v[0.25], 3), round(v[0.75], 3)) for n, v in RESID_Q.items()})

DISTRICT_NAME = META["district_display"]
WARD_NAME = META["ward_display"]
CENTROIDS = {  # tâm gần đúng (lat, lon) của các quận/huyện Hà Nội trước sáp nhập 2025
    "ba dinh": (21.034, 105.814), "hoan kiem": (21.0285, 105.852), "tay ho": (21.070, 105.818),
    "long bien": (21.048, 105.889), "cau giay": (21.033, 105.790), "dong da": (21.013, 105.824),
    "hai ba trung": (21.006, 105.858), "hoang mai": (20.974, 105.864), "thanh xuan": (20.993, 105.811),
    "nam tu liem": (21.012, 105.760), "bac tu liem": (21.070, 105.760), "ha dong": (20.956, 105.757),
    "son tay": (21.138, 105.505), "ba vi": (21.199, 105.423), "phuc tho": (21.109, 105.560),
    "dan phuong": (21.087, 105.670), "hoai duc": (21.025, 105.700), "quoc oai": (20.993, 105.640),
    "thach that": (21.020, 105.550), "chuong my": (20.876, 105.660), "thanh oai": (20.861, 105.770),
    "thuong tin": (20.870, 105.860), "phu xuyen": (20.740, 105.910), "ung hoa": (20.730, 105.770),
    "my duc": (20.680, 105.740), "thanh tri": (20.942, 105.845), "gia lam": (21.028, 105.950),
    "dong anh": (21.136, 105.850), "soc son": (21.256, 105.849), "me linh": (21.180, 105.720),
}
DSTAT = (TRAIN.groupby("district_key")["unit_price"].agg(["median", "size"])
              .rename(columns={"median": "don_gia", "size": "so_tin"}).reset_index())
DSTAT["ten"] = DSTAT["district_key"].map(DISTRICT_NAME).fillna(DSTAT["district_key"])
DSTAT["lat"] = DSTAT["district_key"].map(lambda k: CENTROIDS.get(k, (np.nan, np.nan))[0])
DSTAT["lon"] = DSTAT["district_key"].map(lambda k: CENTROIDS.get(k, (np.nan, np.nan))[1])
DSTAT["don_gia_tr"] = DSTAT["don_gia"] / 1e6
DSTAT = DSTAT.dropna(subset=["lat"]).sort_values("don_gia", ascending=False).reset_index(drop=True)
print(f"Bản đồ: {len(DSTAT)} quận/huyện")

## 2. Logic dự đoán, giải thích, trích thông tin từ tin rao

In [ ]:
def _num(x):
    if x is None:
        return np.nan
    try:
        x = float(x)
    except (TypeError, ValueError):
        return np.nan
    return np.nan if math.isnan(x) or x <= 0 else x

def road_type_of(road_width, is_mat_pho):
    if not np.isnan(road_width):
        return "ngo_nho_<3m" if road_width < 3 else ("ngo_oto_3-6m" if road_width < 6 else "duong_lon_>=6m")
    return "duong_lon_>=6m" if is_mat_pho else "unknown"

def make_row(district, ward, area, floors, bedrooms, bathrooms, frontage, road, direction, flags):
    # Tạo 1 dòng DataFrame đúng định dạng đầu vào của pipeline
    area = _num(area)
    flags = set(flags or [])
    row = {
        "log_area": np.log(area) if not np.isnan(area) else np.nan,
        "floor_count": _num(floors), "bedroom_count": _num(bedrooms), "bathroom_count": _num(bathrooms),
        "frontage_width": _num(frontage), "road_width": _num(road),
    }
    for f in BIN:
        row[f] = int(f in flags)
    ward = ward or "unknown"
    row["district_key"] = district
    row["ward_loc"] = f"{district} | {ward}"
    row["house_direction"] = direction or "Không rõ"
    row["road_type"] = road_type_of(row["road_width"], row.get("is_mat_pho", 0) == 1)
    return pd.DataFrame([row])[FEATURES], area

def predict_all(X):
    return {n: float(m.predict(X)[0]) for n, m in MODELS.items()}

GROUP_LABEL = {"log_area": "Diện tích", "floor_count": "Số tầng", "bedroom_count": "Phòng ngủ",
               "bathroom_count": "Phòng vệ sinh", "frontage_width": "Mặt tiền", "road_width": "Độ rộng đường"}

def feature_group(name):
    body = name.split("__", 1)[1]
    if name.startswith("num__missingindicator_"):
        return GROUP_LABEL.get(body.replace("missingindicator_", ""), body)
    if name.startswith("num__"):
        return GROUP_LABEL.get(body, body)
    if name.startswith("bin__"):
        return FLAG_LABELS.get(body, body)
    if body.startswith("district_key"):    return "Quận/huyện"
    if body.startswith("ward_loc"):        return "Phường"
    if body.startswith("house_direction"): return "Hướng nhà"
    if body.startswith("road_type"):       return "Loại đường/ngõ"
    return body

def explain(model_name, X):
    # Đóng góp (theo %) của từng yếu tố so với 'căn nhà trung bình' của tập train
    pipe = MODELS[model_name].regressor_
    pre, reg = pipe.named_steps["pre"], pipe.named_steps["reg"]
    z = np.asarray(pre.transform(X), dtype=float)[0]
    names = pre.get_feature_names_out()
    # Căn tham chiếu: kích thước & vị trí trung bình, KHÔNG có tiện ích nào (cờ = 0)
    # → tiện ích chưa tích sẽ không hiện ra như một tác động âm
    zref = ZBAR[model_name].copy()
    zref[np.char.startswith(names.astype(str), "bin__")] = 0.0
    contrib = reg.coef_ * (z - zref)
    base = float(np.exp(reg.intercept_ + reg.coef_ @ zref))
    groups = pd.Series(contrib, index=[feature_group(n) for n in names]).groupby(level=0).sum()
    df = pd.DataFrame({"yếu tố": groups.index, "log": groups.values})
    df["tác động (%)"] = (np.exp(df["log"]) - 1) * 100
    df = df[df["log"].abs() > 1e-4].sort_values("log", key=np.abs, ascending=False)
    return base, df.reset_index(drop=True)

def interval(model_name, price):
    q = RESID_Q[model_name]
    return (price * math.exp(q[0.25]), price * math.exp(q[0.75]),
            price * math.exp(q[0.10]), price * math.exp(q[0.90]))

def what_if(model_name, args):
    # Thay đổi từng yếu tố một → giá đổi bao nhiêu %
    district, ward, area, floors, bedrooms, bathrooms, frontage, road, direction, flags = args
    flags = list(flags or [])
    X0, _ = make_row(*args)
    p0 = MODELS[model_name].predict(X0)[0]
    scen = []
    if not np.isnan(_num(area)):
        scen.append(("Diện tích +10 m²", (district, ward, _num(area) + 10, floors, bedrooms, bathrooms, frontage, road, direction, flags)))
    if not np.isnan(_num(floors)):
        scen.append(("Thêm 1 tầng", (district, ward, area, _num(floors) + 1, bedrooms, bathrooms, frontage, road, direction, flags)))
    if not np.isnan(_num(frontage)):
        scen.append(("Mặt tiền +1 m", (district, ward, area, floors, bedrooms, bathrooms, _num(frontage) + 1, road, direction, flags)))
    for f in ["thang_may", "car_access", "kinh_doanh", "is_mat_pho", "lo_goc", "has_so_do"]:
        if f not in flags:
            scen.append((f"Thêm: {FLAG_LABELS[f]}", (district, ward, area, floors, bedrooms, bathrooms, frontage, road, direction, flags + [f])))
    rows = []
    for label, a in scen:
        X1, _ = make_row(*a)
        p1 = MODELS[model_name].predict(X1)[0]
        rows.append({"thay đổi": label, "giá mới (tỷ)": round(p1 / 1e9, 2), "chênh lệch": f"{(p1 / p0 - 1) * 100:+.1f}%"})
    return pd.DataFrame(rows)

def range_warnings(area, floors, bedrooms, bathrooms, frontage, road):
    st, out = META["numeric_stats"], []
    for key, val, lab in [("area", area, "Diện tích"), ("floor_count", floors, "Số tầng"),
                          ("bedroom_count", bedrooms, "Phòng ngủ"), ("bathroom_count", bathrooms, "Phòng vệ sinh"),
                          ("frontage_width", frontage, "Mặt tiền"), ("road_width", road, "Độ rộng đường")]:
        v = _num(val)
        if not np.isnan(v) and not (st[key]["p1"] <= v <= st[key]["p99"]):
            out.append(f"{lab} = {v:g} nằm ngoài khoảng phổ biến của dữ liệu ({st[key]['p1']:g}–{st[key]['p99']:g})")
    return out

# ---------- Trích thông tin từ nguyên văn tin rao ----------
TEXT_FLAGS = {
    "has_so_do":  r"sổ\s*(?:đỏ|hồng|riêng|chính\s*chủ|vuông|nét|sẵn|đẹp)|sđcc|\bsdcc\b|so\s*hong|so\s*do\s*chinh\s*chu",
    "car_access": r"(?:ô\s*tô|ôtô|oto|xe\s*hơi)\s*(?:\d+\s*chỗ\s*)?(?:vào|đỗ|đậu|tránh|chạy|lùi|ngủ|qua|vao|do\b|tranh)"
                  r"|ngõ\s*(?:ô\s*tô|ôtô|oto)|ngo\s*oto|\bgara\b|garage",
    "is_mat_pho": r"mặt\s*(?:phố|đường)|\bmat\s*pho\b|\bmat\s*duong\b",
    "kinh_doanh": r"kinh\s*doanh|buôn\s*bán|vỉa\s*hè|\bvia\s*he\b|\bkd\b",
    "lo_goc":     r"lô\s*góc|căn\s*góc|\blo\s*goc\b|hai\s*mặt\s*(?:thoáng|tiền|ngõ)|2\s*mặt\s*(?:thoáng|tiền|ngõ)",
    "no_hau":     r"nở\s*hậu|\bno\s*hau\b",
    "thang_may":  r"thang\s*máy|\bthang\s*may\b",
}

def strip_accents(s):
    s = unicodedata.normalize("NFD", s)
    return "".join(c for c in s if unicodedata.category(c) != "Mn").replace("đ", "d").replace("Đ", "D")

def norm_key(s):
    return re.sub(r"[^a-z0-9]+", " ", strip_accents(str(s)).lower()).strip()

def _f(s):
    return float(s.replace(",", "."))

def _first(pattern, text):
    m = re.search(pattern, text)
    return _f(m.group(1)) if m else None

def parse_listing(text):
    t = unicodedata.normalize("NFC", text or "").lower()
    k = " " + norm_key(text or "") + " "
    out = {
        "area": _first(r"(?:diện\s*tích|dt|dtsd|dtđ)\s*[:=]?\s*(\d+(?:[.,]\d+)?)", t) or _first(r"(\d+(?:[.,]\d+)?)\s*(?:m2|m²|met vuong|mét vuông)", t),
        "floors": _first(r"(\d+(?:[.,]5)?)\s*(?:tầng|tang\b|lầu)", t),
        "bedrooms": _first(r"(\d+)\s*(?:pn\b|phòng\s*ngủ|ngủ)", t),
        "bathrooms": _first(r"(\d+)\s*(?:wc\b|vs\b|vệ\s*sinh|toilet|nhà\s*tắm)", t),
        "frontage": _first(r"(?:mặt\s*tiền|\bmt)\s*[:=]?\s*(\d+(?:[.,]\d+)?)\s*m", t),
        "road": _first(r"(?:ngõ|ngách|hẻm|đường)\s*(?:rộng\s*|trước\s*nhà\s*)?(\d+(?:[.,]\d+)?)\s*m\b", t),
    }
    m = re.search(r"hướng\s*(đông\s*nam|đông\s*bắc|tây\s*nam|tây\s*bắc|đông|tây|nam|bắc)", t)
    out["direction"] = " ".join(w.capitalize() for w in m.group(1).split()) if m else None
    out["flags"] = [f for f, p in TEXT_FLAGS.items() if re.search(p, t)]
    # quận: khớp tên dài nhất xuất hiện trong văn bản đã bỏ dấu
    dist = None
    for key in sorted(DISTRICT_NAME, key=len, reverse=True):
        if f" {key} " in k:
            dist = key
            break
    out["district"] = dist
    ward = None
    if dist:
        for w in sorted(META["wards_by_district"].get(dist, []), key=len, reverse=True):
            if re.search(rf"(?:phuong|p|xa|thi tran)\s+{re.escape(w)}\b", k) or f" {w} " in k.replace(f" {dist} ", " "):
                ward = w
                break
    out["ward"] = ward
    return out

def random_test_row(seed=None):
    i = np.random.default_rng(seed).integers(len(TEST))
    return TEST.iloc[i]

# ---------- kiểm tra nhanh ----------
_X, _a = make_row("dong da", "o cho dua", 45, 5, 4, 4, 4.0, None, "Đông Nam", ["has_so_do", "car_access"])
_p = predict_all(_X)
print({k: round(v / 1e9, 2) for k, v in _p.items()})
_b, _e = explain("Lasso", _X)
print("Giá nền:", round(_b / 1e9, 2), "tỷ"); print(_e.head(8).round(2))
_chk = _b * np.exp(_e["log"].sum())
assert abs(_chk / _p["Lasso"] - 1) < 0.02, (_chk, _p["Lasso"])
print(parse_listing("Bán nhà ngõ 3m phố Kim Mã, phường Ngọc Khánh, Ba Đình. DT 42m2, 5 tầng, mặt tiền 4,2m, 4PN 4WC, sổ đỏ chính chủ, hướng Đông Nam, ô tô đỗ cửa."))

## 3. Biểu đồ

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

INK, MUTED, ACCENT, POS, NEG = "#1f2937", "#6b7280", "#334155", "#475569", "#b45309"
LAYOUT = dict(template="plotly_white", font=dict(family="Inter, Arial, sans-serif", size=13, color=INK),
              margin=dict(l=10, r=10, t=40, b=10))

def fig_explain(base, df, pred):
    d = df.head(10).iloc[::-1]
    colors = [POS if v >= 0 else NEG for v in d["tác động (%)"]]
    fig = go.Figure(go.Bar(x=d["tác động (%)"], y=d["yếu tố"], orientation="h", marker_color=colors,
                           text=[f"{v:+.1f}%" for v in d["tác động (%)"]], textposition="outside",
                           hovertemplate="%{y}: %{x:+.1f}%<extra></extra>"))
    fig.update_layout(**LAYOUT, height=380,
                      title=dict(text=f"Căn tham chiếu {base/1e9:.1f} tỷ (diện tích, vị trí trung bình, chưa có tiện ích) → {pred/1e9:.1f} tỷ",
                                 font=dict(size=13)),
                      xaxis=dict(title="tác động lên giá (%)", zeroline=True, zerolinecolor="#9ca3af", ticksuffix="%"),
                      yaxis=dict(title=None))
    return fig

def fig_map(selected=None, height=460, zoom=9.2):
    fig = px.scatter_mapbox(DSTAT, lat="lat", lon="lon", size="so_tin", color="don_gia_tr",
                            color_continuous_scale=["#e5e7eb", "#94a3b8", "#334155"], size_max=34,
                            hover_name="ten", hover_data={"don_gia_tr": ":.0f", "so_tin": ":,", "lat": False, "lon": False},
                            labels={"don_gia_tr": "triệu/m²", "so_tin": "số tin"}, zoom=zoom,
                            center={"lat": 21.0, "lon": 105.72}, height=height)
    fig.update_layout(mapbox_style="carto-positron", margin=dict(l=0, r=0, t=0, b=0),
                      coloraxis_colorbar=dict(title="Đơn giá<br>trung vị<br>(tr/m²)", thickness=12),
                      font=dict(family="Inter, Arial, sans-serif"))
    if selected in CENTROIDS:
        lat, lon = CENTROIDS[selected]
        fig.add_trace(go.Scattermapbox(lat=[lat], lon=[lon], mode="markers+text",
                                       marker=dict(size=18, color=NEG, opacity=0.9),
                                       text=[DISTRICT_NAME.get(selected, selected)], textposition="top right",
                                       textfont=dict(size=14, color=INK), hoverinfo="skip", showlegend=False))
    return fig

def fig_compare(preds, actual=None):
    names = list(preds)
    fig = go.Figure(go.Bar(x=[preds[n] / 1e9 for n in names], y=names, orientation="h", marker_color="#94a3b8",
                           text=[f"{preds[n]/1e9:.2f}" for n in names], textposition="outside",
                           hovertemplate="%{y}: %{x:.2f} tỷ<extra></extra>"))
    if actual is not None:
        fig.add_vline(x=actual / 1e9, line_dash="dash", line_color=NEG,
                      annotation_text=f"Giá rao thật: {actual/1e9:.2f} tỷ", annotation_position="top")
    fig.update_layout(**LAYOUT, height=300, xaxis_title="tỷ đồng", yaxis=dict(autorange="reversed"))
    return fig

## 4. Giao diện

In [ ]:
import gradio as gr

DISTRICT_CHOICES = sorted([(v, k) for k, v in DISTRICT_NAME.items()], key=lambda x: x[0])
DIRECTIONS = META["house_directions"]
FLAG_CHOICES = [(v, k) for k, v in FLAG_LABELS.items()]
EXPLAIN_MODELS = [n for n in ["Lasso", "Elastic Net", "Ridge", "Linear Regression", "Lasso (1-SE)"] if n in LINEAR]
PRED_MODELS = list(MODELS)
DEFAULT_PRED = "Gradient Boosting" if "Gradient Boosting" in MODELS else EXPLAIN_MODELS[0]
METRIC = {r["mô hình"]: r for r in META["test_metrics"]}

def ward_choices(district):
    ws = META["wards_by_district"].get(district, [])
    return [("Không rõ / phường khác", "unknown")] + [(WARD_NAME.get(w, w.title()), w) for w in ws]

def on_district(district, current_ward):
    # giữ phường đang chọn nếu thuộc quận mới (khi căn mẫu / tự điền đặt cả quận lẫn phường)
    choices = ward_choices(district)
    valid = {v for _, v in choices}
    return gr.update(choices=choices, value=current_ward if current_ward in valid else "unknown")

PRESETS = {
    "Nhà ngõ Đống Đa · 40 m² · 5 tầng": ("dong da", "o cho dua", 40, 5, 4, 4, 3.8, 2.5, "Không rõ", ["has_so_do"]),
    "Nhà ô tô đỗ cửa Cầu Giấy · 60 m²": ("cau giay", "dich vong", 60, 6, 5, 5, 5, 5, "Đông Nam", ["has_so_do", "car_access", "thang_may"]),
    "Mặt phố kinh doanh Hoàn Kiếm · 55 m²": ("hoan kiem", "hang bac", 55, 5, 4, 5, 4.5, 10, "Không rõ", ["is_mat_pho", "kinh_doanh", "has_so_do"]),
    "Nhà Hà Đông · 35 m² · ngõ nhỏ": ("ha dong", "unknown", 35, 4, 3, 3, None, 2, "Tây", ["has_so_do"]),
}

def load_preset(name):
    d, w, a, f, b, t, fr, r, di, fl = PRESETS[name]
    choices = ward_choices(d)
    valid = {v for _, v in choices}
    return (gr.update(value=d), gr.update(choices=choices, value=w if w in valid else "unknown"),
            a, f, b, t, fr, r, di, fl)

def fmt_ty(v):
    return f"{v/1e9:,.2f}".replace(",", "X").replace(".", ",").replace("X", ".") + " tỷ"

def price_card(model_name, price, area, warns):
    lo50, hi50, lo80, hi80 = interval(model_name, price)
    unit = f"{price / area / 1e6:,.0f} triệu/m²".replace(",", ".") if not np.isnan(area) else "—"
    w = "".join(f"<div class='warn'>⚠ {x}</div>" for x in warns)
    return f'''
    <div class="card">
      <div class="label">Giá dự đoán · {model_name}</div>
      <div class="price">{fmt_ty(price)}</div>
      <div class="sub">Đơn giá ≈ {unit}</div>
      <div class="range"><b>Khoảng tham khảo:</b> {fmt_ty(lo50)} – {fmt_ty(hi50)} <span class="muted">(50% tin thật rơi vào)</span><br>
      {fmt_ty(lo80)} – {fmt_ty(hi80)} <span class="muted">(80%)</span></div>
      {w}
      <div class="note">Giá <b>rao bán</b> ước tính từ ~390 nghìn tin đăng 6/2025–3/2026, không phải giá giao dịch. MAPE của mô hình này trên tập test: {METRIC.get(model_name, {}).get("MAPE (%)", float("nan")):.1f}%.</div>
    </div>'''

INPUT_KEYS = ["district", "ward", "area", "floors", "bedrooms", "bathrooms", "frontage", "road", "direction", "flags"]

def run(pred_model, model_name, district, ward, area, floors, bedrooms, bathrooms, frontage, road, direction, flags):
    if not district:
        raise gr.Error("Hãy chọn quận/huyện.")
    if np.isnan(_num(area)):
        raise gr.Error("Hãy nhập diện tích.")
    args = (district, ward, area, floors, bedrooms, bathrooms, frontage, road, direction, flags)
    X, a = make_row(*args)
    preds = predict_all(X)
    base, ex = explain(model_name, X)
    card = price_card(pred_model, preds[pred_model], a, range_warnings(area, floors, bedrooms, bathrooms, frontage, road))
    tbl = pd.DataFrame({"mô hình": list(preds), "giá dự đoán (tỷ)": [round(v / 1e9, 2) for v in preds.values()],
                        "MAPE test (%)": [round(METRIC[n]["MAPE (%)"], 1) if n in METRIC else None for n in preds]})
    # "Nếu… thì sao" dùng mô hình tuyến tính: tác động biên mượt & nhất quán (mô hình cây có thể nhảy bậc, khó diễn giải)
    return card, tbl, fig_explain(base, ex, preds[model_name]), what_if(model_name, args), fig_map(district)

def autofill(text):
    p = parse_listing(text)
    found = [lab for lab, key in [("quận", "district"), ("phường", "ward"), ("diện tích", "area"), ("số tầng", "floors"),
                                  ("phòng ngủ", "bedrooms"), ("WC", "bathrooms"), ("mặt tiền", "frontage"),
                                  ("độ rộng ngõ/đường", "road"), ("hướng", "direction")] if p[key]]
    found += [FLAG_LABELS[f] for f in p["flags"]]
    msg = ("**Đã nhận diện:** " + ", ".join(found)) if found else "Không nhận diện được thông tin nào — hãy nhập tay."
    keep = gr.update()
    d = p["district"]
    return (msg,
            gr.update(value=d) if d else keep,
            gr.update(choices=ward_choices(d), value=p["ward"] or "unknown") if d else keep,
            *[p[k] if p[k] is not None else keep for k in ["area", "floors", "bedrooms", "bathrooms", "frontage", "road"]],
            p["direction"] if p["direction"] in DIRECTIONS else keep,
            p["flags"])

LABELS_VI = {"district": "Quận/huyện", "ward": "Phường", "area": "Diện tích (m²)", "floor_count": "Số tầng",
             "bedroom_count": "Phòng ngủ", "bathroom_count": "Phòng vệ sinh", "frontage_width": "Mặt tiền (m)",
             "road_width": "Độ rộng đường (m)", "house_direction": "Hướng", "title": "Tiêu đề tin"}

def test_listing():
    r = random_test_row()
    X = r[FEATURES].to_frame().T.astype({c: float for c in NUM})
    preds = predict_all(X)
    info = {LABELS_VI[c]: r[c] for c in ["title", "district", "ward", "area", "floor_count", "bedroom_count",
                                         "bathroom_count", "frontage_width", "road_width", "house_direction"] if c in r.index}
    info["Tiện ích"] = ", ".join(FLAG_LABELS[f] for f in BIN if r[f] == 1) or "—"
    info_df = pd.DataFrame({"thuộc tính": list(info), "giá trị": ["—" if pd.isna(v) else v for v in info.values()]})
    actual = float(r["price"])
    err = pd.DataFrame({"mô hình": list(preds), "dự đoán (tỷ)": [round(v / 1e9, 2) for v in preds.values()],
                        "sai lệch": [f"{(v / actual - 1) * 100:+.1f}%" for v in preds.values()]})
    head = f"### Giá rao thật: **{fmt_ty(actual)}**  ·  {r['area']:g} m²  ·  {DISTRICT_NAME.get(r['district_key'], r['district_key'])}"
    return head, info_df, err, fig_compare(preds, actual)

metrics_df = pd.DataFrame(META["test_metrics"]).reindex(columns=["mô hình", "MAE (tỷ)", "RMSE (tỷ)", "MAPE (%)", "MdAPE (%)", "R² log", "số hệ số ≠ 0"]).round(3)
FIG_CAPTIONS = {"01": "Lỗi kiểm định chéo theo α", "02": "Coefficient path: Lasso · Elastic Net · Ridge",
                "03": "Lasso: số đặc trưng giữ lại theo α", "04": "Phòng ngủ & WC dưới L1 / Elastic Net",
                "05": "Random Forest: mức quan trọng đặc trưng", "06": "So sánh mô hình", "07": "Dự đoán vs thực tế",
                "08": "MAPE theo phân khúc giá", "09": "Thí nghiệm kích thước dữ liệu"}
GALLERY = [(p, FIG_CAPTIONS.get(os.path.basename(p)[:2], os.path.basename(p))) for p in sorted(glob.glob(os.path.join(FIG_DIR, "*.png")))]
CAP4 = {"so_sanh_cuoi": "So sánh cuối: CV vs test", "ablation": "Ablation nhóm đặc trưng", "phan_du_lasso": "Phần dư — Lasso",
        "phan_du_hgb": "Phần dư — Gradient Boosting", "don_gia_theo_thang": "Đơn giá theo tháng đăng"}
GALLERY += [(p, CAP4.get(os.path.basename(p)[:-4], os.path.basename(p))) for p in sorted(glob.glob(os.path.join(FIG_DIR4, "*.png")))]
map_table = DSTAT[["ten", "don_gia_tr", "so_tin"]].rename(columns={"ten": "quận/huyện", "don_gia_tr": "đơn giá trung vị (tr/m²)", "so_tin": "số tin"}).round(0)

CSS = '''
.gradio-container {max-width: 1280px !important; margin: auto;}
.card {border: 1px solid #e5e7eb; border-radius: 10px; padding: 18px 20px; background: #fafafa;}
.card .label {color: #6b7280; font-size: 13px; text-transform: uppercase; letter-spacing: .04em;}
.card .price {font-size: 40px; font-weight: 700; color: #111827; margin: 4px 0;}
.card .sub {color: #374151; font-size: 15px; margin-bottom: 10px;}
.card .range {color: #374151; font-size: 14px; line-height: 1.6;}
.card .muted, .card .note {color: #6b7280;}
.card .note {font-size: 12px; margin-top: 12px; border-top: 1px solid #e5e7eb; padding-top: 8px;}
.card .warn {color: #92400e; background: #fef3c7; border-radius: 6px; padding: 6px 10px; margin-top: 8px; font-size: 13px;}
h1 {font-weight: 700 !important;}
'''
theme = gr.themes.Base(primary_hue=gr.themes.colors.slate, secondary_hue=gr.themes.colors.slate,
                       neutral_hue=gr.themes.colors.gray, radius_size=gr.themes.sizes.radius_sm,
                       font=[gr.themes.GoogleFont("Inter"), "Arial", "sans-serif"])

with gr.Blocks(theme=theme, css=CSS, title="Định giá nhà Hà Nội") as demo:
    gr.Markdown("# Định giá nhà Hà Nội\nDự đoán giá rao bán nhà riêng/nhà phố Hà Nội bằng các mô hình hồi quy có chính quy hoá (Ridge, Lasso, Elastic Net) và mô hình cây (Random Forest, Gradient Boosting).")
    with gr.Tab("Định giá"):
        with gr.Row():
            with gr.Column(scale=5):
                with gr.Accordion("Dán nguyên văn tin rao để tự điền (tuỳ chọn)", open=False):
                    listing = gr.Textbox(lines=4, show_label=False,
                                         placeholder="VD: Bán nhà ngõ 3m phố Kim Mã, Ba Đình. DT 42m2, 5 tầng, mặt tiền 4,2m, 4PN 4WC, sổ đỏ, ô tô đỗ cửa…")
                    with gr.Row():
                        fill_btn = gr.Button("Tự điền", size="sm")
                    fill_msg = gr.Markdown()
                preset = gr.Dropdown(list(PRESETS), label="Hoặc chọn căn mẫu", value=None)
                with gr.Row():
                    district = gr.Dropdown(DISTRICT_CHOICES, label="Quận/huyện", value="dong da")
                    ward = gr.Dropdown(ward_choices("dong da"), label="Phường", value="unknown")
                with gr.Row():
                    area = gr.Number(label="Diện tích (m²)", value=45, minimum=0)
                    floors = gr.Number(label="Số tầng", value=5, minimum=0)
                with gr.Row():
                    bedrooms = gr.Number(label="Phòng ngủ", value=4, minimum=0)
                    bathrooms = gr.Number(label="Phòng vệ sinh", value=4, minimum=0)
                with gr.Row():
                    frontage = gr.Number(label="Mặt tiền (m) — để trống nếu không rõ", value=None, minimum=0)
                    road = gr.Number(label="Độ rộng ngõ/đường (m) — để trống nếu không rõ", value=None, minimum=0)
                direction = gr.Dropdown(DIRECTIONS, label="Hướng nhà", value="Không rõ")
                flags = gr.CheckboxGroup(FLAG_CHOICES, label="Đặc điểm", value=["has_so_do"])
                with gr.Row():
                    pred_model = gr.Dropdown(PRED_MODELS, label="Mô hình dự đoán", value=DEFAULT_PRED)
                    model_name = gr.Dropdown(EXPLAIN_MODELS, label="Mô hình giải thích (tuyến tính)", value=EXPLAIN_MODELS[0])
                go_btn = gr.Button("Định giá", variant="primary")
            with gr.Column(scale=7):
                card = gr.HTML()
                with gr.Tabs():
                    with gr.Tab("Vì sao ra giá này?"):
                        explain_plot = gr.Plot(show_label=False)
                    with gr.Tab("Nếu… thì sao?"):
                        gr.Markdown("Tính theo **mô hình giải thích (tuyến tính)** để tác động của từng thay đổi nhất quán.")
                        whatif_tbl = gr.Dataframe(interactive=False, wrap=True)
                    with gr.Tab("So sánh 7 mô hình"):
                        models_tbl = gr.Dataframe(interactive=False)
                    with gr.Tab("Vị trí"):
                        map_small = gr.Plot(show_label=False)
        inputs = [pred_model, model_name, district, ward, area, floors, bedrooms, bathrooms, frontage, road, direction, flags]
        outputs = [card, models_tbl, explain_plot, whatif_tbl, map_small]
        go_btn.click(run, inputs, outputs)
        district.change(on_district, [district, ward], ward)
        preset.change(load_preset, preset, [district, ward, area, floors, bedrooms, bathrooms, frontage, road, direction, flags]) \
              .then(run, inputs, outputs)
        fill_btn.click(autofill, listing, [fill_msg, district, ward, area, floors, bedrooms, bathrooms, frontage, road, direction, flags])

    with gr.Tab("Kiểm tra với tin thật"):
        gr.Markdown("Lấy ngẫu nhiên một tin **trong tập test** (mô hình chưa từng thấy) và so giá rao thật với dự đoán của từng mô hình.")
        rnd_btn = gr.Button("Lấy tin ngẫu nhiên", variant="primary")
        t_head = gr.Markdown()
        with gr.Row():
            t_info = gr.Dataframe(interactive=False, wrap=True)
            with gr.Column():
                t_err = gr.Dataframe(interactive=False)
                t_plot = gr.Plot(show_label=False)
        rnd_btn.click(test_listing, None, [t_head, t_info, t_err, t_plot])

    with gr.Tab("Bản đồ giá"):
        gr.Markdown("Đơn giá rao bán **trung vị** theo quận/huyện (kích thước điểm = số tin). Vị trí là tâm gần đúng của quận/huyện theo địa giới trước sáp nhập 2025.")
        with gr.Row():
            with gr.Column(scale=3):
                gr.Plot(value=fig_map(height=620, zoom=9.4), show_label=False)
            with gr.Column(scale=2):
                gr.Dataframe(map_table, interactive=False, max_height=620)

    with gr.Tab("Kết quả mô hình"):
        gr.Markdown(f"Đánh giá trên tập test {META['test_size']:,} tin (train {META['train_size']:,} tin).")
        gr.Dataframe(metrics_df, interactive=False)
        gr.Gallery(GALLERY, columns=3, height="auto", object_fit="contain", label="Biểu đồ từ notebook 02")

    demo.load(run, inputs, outputs)

In [ ]:
demo.launch(share=True, debug=False)

### Ghi chú
- Link `…gradio.live` chỉ hoạt động khi notebook này đang chạy. Dừng demo: **Runtime → Interrupt execution**, hoặc `demo.close()`.
- Chạy trên máy (không cần Colab): cài `gradio`, `plotly`, `scikit-learn=={phiên bản trong demo_meta.json}`, `pandas`, `pyarrow`; đặt `PROJECT_DIR` trỏ tới thư mục `ML_house_price`; chạy các ô như trên.